# Factorized CFA — PDF inference and less-frequent citation ratio

This notebook keeps the **same PDF extraction and target-aware citation construction logic** as the monolithic pipeline, but replaces the monolithic 12-class LLaMA prediction with:

1. **BERT sentiment** → `Negative / Neutral / Positive`
2. **LLaMA-3-8B function classifier** → 4 citation functions
3. deterministic combination → the same **12 final CFA labels**
4. the same locked set of **8 less-frequent / rare labels**

### Added in this version: function-confidence diagnostics

The original **generation-based `Function_Label` is preserved** so the inference definition does not change. In addition, the notebook scores all four allowed function labels by teacher-forced sequence likelihood, length-normalizes their log-likelihoods, and applies a softmax across the four candidate scores. The detail CSVs therefore also contain:

- `Function_Confidence`: normalized candidate probability assigned to the generated `Function_Label`;
- `Prob_Function_Background`, `Prob_Function_Comparison`, `Prob_Function_Using`, `Prob_Function_Extending`;
- `Function_Margin`: difference between the highest and second-highest candidate probabilities;
- `Function_Scored_Top_Label` and `Function_Score_Agreement` for auditing generation-vs-scoring agreement.

These function probabilities are **relative model-confidence scores over the four allowed labels, not calibrated probabilities**. They are diagnostic fields and do not change the primary LFCR calculation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
""" Εγκατάσταση απαραίτητων στοιχείων και βιβλιοθηκών """
!pip install PyMuPDF
!pip install spacy
!pip install sentence-transformers
!pip install torch
!python -m spacy download en_core_web_sm
!pip install -U bitsandbytes
!pip install -U peft transformers accelerate

In [ ]:
# Εισαγωγή του Access Token απο την Hugging face.
import os
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Set HF_TOKEN securely before running this notebook.")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
import re
import gc
import torch
import spacy
#import fitz
import pandas as pd
import numpy as np
import pymupdf

from spacy.symbols import ORTH
from sentence_transformers import SentenceTransformer, util
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

nlp = spacy.load("en_core_web_sm")
nlp.tokenizer.add_special_case("et al.", [{ORTH: "et al."}])

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / (1024 ** 3))
else:
    print("WARNING: CUDA GPU is not available.")

## 1. PDF preprocessing and sentence helpers — preserved from monolithic pipeline

In [ ]:
# Λήψη του PDF από το PATH και μετατροπή σε κείμενο.
def pdf_to_text(pdf_path):
    document = pymupdf.open(pdf_path)
    text = ''

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()

    document.close()

    # Αφαίρεση κειμένου μετά το References
    pattern = r"(.*)(references.*?)$"
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)

    if match:
        final_text = match.group(1)
    else:
        final_text = text

    # Προ-επεξεργασία
    final_text = final_text.replace("–", "-")
    final_text = re.sub(r"(\w+)-\s+(\w+)", r"\1\2", final_text)
    final_text = re.sub(r"(\d+)\s*-\s*(\d+)", r"\1-\2", final_text)

    # το Gururangan et al (2018) το μετατρέπει σε citation μορφής (Gururangan et al., 2018).
    final_text = re.sub(
        r"(\b\w+\s+et al)\.?,?\s*\(\s*(\d{4})\s*\)",
        r"(\1., \2)",
        final_text
    )

    final_text = re.sub(r"(\w+)\s+\((\d{4})\)", r"(\1, \2)", final_text)
    final_text = re.sub(r'^\d+$', '', final_text, flags=re.MULTILINE)

    # Διόρθωση περιπτώσεων τύπου "et al. ."
    final_text = re.sub(
        r"\bet al\.\s*\.",
        "et al.",
        final_text,
        flags=re.IGNORECASE
    )

    return final_text

# Χρήση της βιβλιοθήκης spaCy για τον διαχωρισμό του κειμένου σε προτάσεις.
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

# Ενώνει "ορφανές" προτάσεις που είναι μόνο citation markers (π.χ. [19].) με το πραγματικό κείμενο γύρω τους, ώστε να ανακτηθεί πλήρες citation context.
def merge_citation_only_sentences(sentences):
    merged = []
    i = 0

    citation_only_pattern = re.compile(
        r"""
        ^\s*
        [\[\(]
        \s*
        \d+
        (
            \s*
            (?:,|-)\s*
            \d+
        )*
        \s*
        [\]\)]
        [\]\)]*
        \s*
        [\.,;:–-]*
        \s*
        $
        """,
        re.VERBOSE
    )

    while i < len(sentences):
        sent = sentences[i].strip()

        if citation_only_pattern.match(sent):
            if merged:
                merged[-1] = merged[-1].rstrip() + " " + sent
            elif i + 1 < len(sentences):
                merged.append(sent + " " + sentences[i + 1])
                i += 1
        else:
            merged.append(sent)

        i += 1

    return merged

# new
def merge_sentences_starting_with_bracket_citation(sentences):
    merged = []

    leading_bracket_pattern = re.compile(
        r'^\s*["“”\'‘’]?\s*\[\s*\d+(?:\s*[-,]\s*\d+)*\s*\]\s*[,;:]?'
    )

    for sent in sentences:
        sent = sent.strip()

        if leading_bracket_pattern.match(sent) and merged:
            merged[-1] = merged[-1].rstrip() + " " + sent.lstrip()
        else:
            merged.append(sent)

    return merged

## 2. Citation-context detection — preserved

In [ ]:
def contains_citation(sentence, model, tokenizer, use_special_style=0):

    sentence = str(sentence).strip()

    # Κόβει reference-entry μορφές: Technol. 10, 2, Article 12 (January 2019), 19 pages.
    if re.search(
        r"\b[A-Z][A-Za-z.&\s]*\.\s*\d+\s*,\s*\d+\s*,\s*Article\s+\d+\s*"
        r"\([A-Za-z]+\s+\d{4}\)\s*,\s*\d+\s+pages?\b",
        sentence,
        re.IGNORECASE
    ):
        return False

    # ΝΝ
    # Κόβει μόνο journal header/footer metadata, δηλαδή όταν μετά το (Journal, Year) ακολουθεί μορφή volume/page: π.χ. (Biology, 2019) 20:238
    if re.search(
        r"\([A-Z][A-Za-z\s\-]+,\s*\d{4}\)\s*\d+\s*:\s*\d+",
        sentence
    ):
        print("κόπηκε λόγω volume/page")
        return False
    # ΝΝ

    # Κόψε figure panel labels τύπου (A), (B), κλπ.
    if re.search(r"\([A-Z]\)\s*(Expression|Representative|Immunostaining|Staining|Analysis)?", sentence, re.IGNORECASE):
        return False

    # Κόψε biomedical markers που μοιάζουν με superscript citations
    if re.search(r"\b(?:Th\d{1,2}|CD\d{1,3}|S\d{1,3}|IL-\d{1,3}|TGF-?β\d*)\b[,;:]?", sentence):
        return False

    # Κόψε figure/table references, ακόμα κι αν κόπηκαν από το spaCy/PDF extraction
    if re.search(
        r"\(?\s*(Fig\.?|Figure|Table|Supplementary\s+Table|Supplementary\s+Fig\.?)\b",
        sentence,
        re.IGNORECASE
    ):
        return False

    # Κόψε references σε figures/tables, ακόμα και τύπου Fig. 3H, Fig. 3C, Table S7
    if re.search(
        r"\(?\s*\b(?:Fig\.?|Figure|Table|Supplementary\s+(?:Table|Fig\.?|Figure))\b",
        sentence,
        re.IGNORECASE
    ):
        return False


    # Κόψε panel labels τύπου (A), (B), (C)
    if re.match(r"^\s*\([A-Z]\)\s*", sentence):
        return False


    # ενεργοποιούνται για paper με ειδικό στυλ citation superscipt. π.χ. women,23 όπου το 23 είναι εκθέτης
    special_patterns = [
        re.compile(r"\b[A-Z][A-Za-z\-]+\s+et\s+al\.?\d{1,3}\b"),

    # women,23 / overtreatment.26
        re.compile(r"\b[a-z]{4,}[.,]\d{1,3}\b"),

    # overtreatment.26,27 / women,23,24
        re.compile(r"\b[a-z]{4,}[.,]\d{1,3}(?:,\d{1,3})+\b"),

    # features.47-49
        re.compile(r"\b[a-z]{4,}[.,]\d{1,3}\s*[-–]\s*\d{1,3}\b"),

    # humans8,9 / humans8,9,10,12
        re.compile(r"\b[a-z]{4,}\d{1,3}(?:,\d{1,3})+\b"),

    # damage30-33
        re.compile(r"\b[a-z]{4,}\d{1,3}\s*[-–]\s*\d{1,3}\b"),

    # intervention8,13-15
        re.compile(r"\b[a-z]{4,}\d{1,3}(?:\s*,\s*\d{1,3}(?:\s*[-–]\s*\d{1,3})?)+\b")
    ]


    # ενεργοποιούνται για paper με ειδικό στυλ citation παρενθετικο. π.χ. assemblies (4), alone (6,7), molecules (8–11) κλπ.
    parentesis_patterns = [
        #re.compile(r"\((\d+(?:\s*[-–,]\s*\d+)*)\)")
        re.compile(r"\b[a-zA-Z][a-zA-Z\-]{2,}\s*\(\s*\d{1,3}(?:\s*[-–,]\s*\d{1,3})*\s*\)")

    ]


    # Αγνόησε author lines τύπου "S. Azevedo1"
    if re.match(r"^[A-Z]\.\s*[A-Z][a-z]+\d{1,3}$", sentence.strip()):
        return False

    # Αγνόησε ένα ή περισσότερα initials + επώνυμο + αριθμός (superscript affiliation) π.χ. S.Azevedo1, S. A. Azevedo1 κλπ.
    if re.match(r"^(?:[A-Z]\.\s*)+[A-Z][A-Za-z\-]+\d{1,3}$", sentence.strip()):
        return False

    # Κόβει patterns alphanumeric μέσα σε ()
    if re.search(r"\(\s*\d+[A-Za-z]+\s*\)", sentence):
        return False


    # Κόβει patterns τύπου Fig/Table μέσα σε παρένθεση
    if re.search(r"\((?:Fig\.?|Figure|Table|Section|Eq\.?|Supplementary)[^)]*\)", sentence, re.IGNORECASE):
        return False



    # ΚΟΨΙΜΟ ΜΗ-CITATION
    if re.search(r"\b(Table|Figure|Fig\.?|Section|Eq\.?|Equation)\s*\d+\b", sentence, re.IGNORECASE):
        return False

    # SAFE PATTERNS CITATIONS (κλασσικές citations (author, year))
    safe_patterns = [
        # (Macken et al., 2008)
        re.compile(r"\([^\)]*et al\.?,?\s*\d{4}[a-z]?\)"),

        # (Kupiec, 1993) | (Lloyd 1982)
        re.compile(r"\([A-Z][A-Za-z\-]+,?\s*\d{4}[a-z]?\)"),

        # (Frantzi and Ananiadou, 1999)
        re.compile(r"\([A-Z][A-Za-z\-]+\s+and\s+[A-Z][A-Za-z\-]+,\s*\d{4}[a-z]?\)"),

        # Moore and Quirk (2008)
        re.compile(r"\b[A-Z][A-Za-z\-]+\s+and\s+[A-Z][A-Za-z\-]+\s*\(\d{4}[a-z]?\)"),

        # Cer et al (2008) | Cer et al. (2008) | Gururangan et al., (2018) | Gururangan et al., (2018))
        #re.compile(r"\b[A-Z][A-Za-z\-]+\s+et al\.?\s*\(\d{4}[a-z]?\)"),
        re.compile(r"\b[A-Z][A-Za-z\-]+\s+et\s+al\.?,?\s*\(\d{4}[a-z]?\)\)?"),

        # πιάνει (Gläser, 2012) patterns.
        re.compile(r"\([A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ\-]+,?\s*\d{4}[a-z]?\)"),

        # πιάνει μορφές όπως: (Osuna, Cruz-Castro, & Sanz-Menendez, 2011) | (Brown, Taylor, Johnson, 2019).
        re.compile(
            r"\([A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ\-]+"
            r"(?:,\s*[A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ\-]+)*"
            r"(?:,\s*&\s*[A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ\-]+)?"
            r",\s*\d{4}[a-z]?\)"
        ),

        # πιάνει: (Gläser & Laudel, 2007) | (Gläser & Laudel, 2007; Gläser, Lange, Laudel, & Schimank, 2010a) | (Adam et al., 2020; Brown & Smith, 2021)
        re.compile(r"\((?:[A-ZÀ-ÖØ-Þ][^()]*?\d{4}[a-z]?)(?:;\s*[A-ZÀ-ÖØ-Þ][^()]*?\d{4}[a-z]?)*\)"),

        # πιάνει: models (Airoldi, Blei, Fienberg, & Xing, 2008)
        re.compile(r"\(\s*[A-ZÀ-ÖØ-Þ][^()]*?\d{4}[a-z]?\s*\)")
    ]

    # SAFE PATTERNS CITATIONS (κλασσικές citations με αγκύλες π.χ. [2])
    safe_patterns_2 = [
        # [1], [1,2], [4-7], [4 - 8]
        re.compile(r"\[(\d+(?:\s*[-,]\s*\d+)*)\]"),

        # resources[3], resources [4]
        re.compile(r"\b\w+\s*\[\d+\]"),

        # resources[3-7], resources [4 - 8]
        re.compile(r"\b\w+\s*\[\d+\s*-\s*\d+\]"),

        # resources[1,2,3]
        re.compile(r"\b\w+\s*\[\d+(?:\s*,\s*\d+)+\]"),
    ]

    # αν πρόκειται για paper με ειδικό στυλ citation superscript ενεργοποιώ την λίστα special_patterns.
    # αν πρόκειται για paper με ειδικό στυλ citation - παρενθετικο ενεργοποιώ τα στοιχεία της λίστας parentesis_patterns.
    # αν πρόκειται για paper με κλασσικό στυλ citation (author, year) ενεργοποιώ την λίστα safe_patterns.
    # αν πρόκειται για paper με κλασσικό στυλ citation αγκυλών, ενεργοποιώ την λίστα safe_patterns_2.
    if use_special_style == "1":
        if any(p.search(sentence) for p in special_patterns):
            print("ανιχνεύθηκε pattern superscript")
            return True

    if use_special_style == "2":
        if any(p.search(sentence) for p in parentesis_patterns):
            print("ανιχνεύθηκε pattern parethesis")
            return True

    if use_special_style == "0":
        if any(p.search(sentence) for p in safe_patterns):
            print("ανιχνεύθηκε pattern (author, year)")
            return True
        return False

    if use_special_style == "3":
        if any(p.search(sentence) for p in safe_patterns_2):
            print("ανιχνεύθηκε pattern brackets []")
            return True


    # για use_special_style = "auto"
    if use_special_style == "auto":
        if any(p.search(sentence) for p in safe_patterns):
            print("ανιχνεύθηκε pattern (author, year)")
            return True

        if any(p.search(sentence) for p in safe_patterns_2):
            print("ανιχνεύθηκε pattern brackets []")
            return True

        # ΜΗΝ ενεργοποιείς σκέτο parenthesis style στο auto
        # γιατί πιάνει άσχετα biomedical / figure / panel patterns
        # κράτα το parentesis_patterns μόνο όταν εσύ δώσεις χειροκίνητα
        #if any(p.search(sentence) for p in parentesis_patterns):
        #    print("ανιχνεύθηκε pattern parenthesis ()")
        #    return True

        # Στα PDFs του παρόντος pipeline χρησιμοποιούνται μόνο:
        #   - Author-Year
        #   - IEEE / numeric brackets
        # Τα superscript patterns ΔΕΝ ενεργοποιούνται στο auto mode, ώστε
        # να μειώνονται false positives και unresolved targetization cases.



    has_citation_marker = re.search(r"\[\d+|\(\s*[A-Z][^\)]*\d{4}", sentence)

    if not has_citation_marker:
        return False


    # Αν δεν υπάρχει ξεκάθαρο citation pattern, ΜΗΝ πας στο BERT.
    if use_special_style == "auto":
        return False

    inputs = tokenizer([sentence], return_tensors='pt', padding=True, truncation=True, max_length=128)

    with torch.no_grad():
         outputs = model(**inputs)

    predictions = torch.softmax(outputs.logits, dim=1)
    return predictions[0][1] > 0.8

    # Έλεγχος με κανονικές εκφράσεις (ως πρώτο φίλτρο).
    """if any(pattern.search(sentence) for pattern in patterns):
        return True
    else:
        # Χρήση του μοντέλου αναγνώρισης πλαισίων (ως δεύτερο φίλτρο), για προτάσεις που δεν εντοπίστηκαν με τις κανονικές εκφράσεις.
        inputs = tokenizer([sentence], return_tensors='pt', padding=True, truncation=True, max_length=128)

        with torch.no_grad():
            outputs = model(**inputs)

        predictions = torch.softmax(outputs.logits, dim=1)
        #print(predictions)

        if predictions[0][1] > 0.8:  # Οριο πρόβλεψης 0.8 - Αν η κλάση 1 (1=είναι πλαίσιο παραπομπής) έχει πιθανότητα > 0.8, τότε θεωρείται ότι η sentence είναι πλαίσιο.
            return True
        else:
            return False"""

## 3. Similarity, citation-style mode, and context expansion — preserved

In [ ]:
def calculate_similarity(sent1, sent2, model):
    embedding1 = model.encode(sent1, convert_to_tensor=True)
    embedding2 = model.encode(sent2, convert_to_tensor=True)
    similarity = util.pytorch_cos_sim(embedding1, embedding2)
    return similarity.item()

ep = input("Αν θέλεις χειροκίνητη επεξεργασία αρχείων πάτησε 1. Για αυτόματη πάτησε 2.")
print("στην χειροκίνητη, όλα τα νέα pdf που θα επεξεργαστούν πρέπει να έχουν το ίδιο στυλ αναφορών π.χ. superscript style.")
print("στην αυτόματη, όλα τα νέα pdf που θα επεξεργαστούν μπορεί να έχουν οποιοδήποτε στυλ αναφορών. Ο εντοπισμός του στυλ θα γίνει αυτόματα.")

if ep == "1":
    print("επιλέχτηκε χειροκίνητη επεξεργασία.")
    use_special_style = input("Αν θέλεις να ενεργοποιήσεις κανόνες για κλασσικά citations (author, year) δώσε 0. Αν θέλεις να ενεργοποιήσεις κανόνες για superscript citations δώσε 1. Αν θέλεις να ενεργοποιήσεις κανόνες για παρενθετικά citations δώσε 2. Αν θέλεις να ενεργοποιήσεις κανόνες για citation με αγκύλες δώσε 3. : ")
elif ep == "2":
    print("επιλέχτηκε αυτόματη επεξεργασία.")
    use_special_style = "auto"
else:
    print("Δώσε έγκυρη επιλογή μεταξύ 1 ή 2.")

## 7. Main PDF citation extraction

In [ ]:
def main(pdf_path):
    # PDF → κείμενο
    text = pdf_to_text(pdf_path)

    # Κείμενο → προτάσεις
    sentences = split_sentences(text)
    sentences = merge_citation_only_sentences(sentences)

    #new
    #if use_special_style == "3":
    sentences = merge_sentences_starting_with_bracket_citation(sentences)
    #new

    # Φόρτωση citation context detection μοντέλου BERT.
    tokenizer = BertTokenizer.from_pretrained(
        '/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/pretrained_citation_context_recognition/saved_tokenizer'
    )
    model = BertForSequenceClassification.from_pretrained(
        '/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/pretrained_citation_context_recognition/saved_model'
    )

    # Εντοπισμός citation sentences
    citation_context = [
        sent for sent in sentences
        if contains_citation(sent, model, tokenizer, use_special_style=use_special_style)
    ]

    # SentenceTransformer για επέκταση πλαισίου
    model_S = SentenceTransformer('paraphrase-MiniLM-L6-v2')

    wider_citation_context = set()


    # new
    #leading_bracket_citation = re.compile(
    #    r"^\s*\[\s*\d+(?:\s*[-,]\s*\d+)*\s*\]\s*[,;:]?"
    #)
    # new

    for citation in citation_context:
        index = sentences.index(citation)
        added = False


        # new
        # Αν η πρόταση ξεκινά με [23], [41], [1,2], [4-7] κλπ.,
        # τότε πιθανότατα το spaCy/PDF split έκοψε την πρόταση στη μέση.
        # Άρα την ενώνουμε απευθείας με την προηγούμενη πρόταση.
        #if index > 0 and leading_bracket_citation.match(citation):
        #    prev_sentence = sentences[index - 1]
        #    wider_citation_context.add(prev_sentence.rstrip() + " " + citation.lstrip())
        #    added = True
        #    continue
        # new

        if index > 0:
            prev_sentence = sentences[index - 1]
            if calculate_similarity(citation, prev_sentence, model_S) > 0.6:
                wider_citation_context.add(prev_sentence.rstrip() + ' ' + citation.lstrip())
                added = True

        if index < len(sentences) - 1:
            next_sentence = sentences[index + 1]
            if calculate_similarity(citation, next_sentence, model_S) > 0.6:
                wider_citation_context.add(citation.rstrip() + ' ' + next_sentence.lstrip())
                added = True

        if not added:
            wider_citation_context.add(citation)

    # Επιστρέφουμε ΛΙΣΤΑ (όχι set)
    return list(wider_citation_context)

## 4. Target-aware citation construction — preserved

In [ ]:
TARGET_MARKER = "[TARGET_CITATION]"
OTHER_MARKER = "[OTHER_CITATION]"

def _expand_ieee_cluster(cluster_text):
    """
    Μετατρέπει ένα IEEE cluster σε λίστα μεμονωμένων citations.

    Παραδείγματα:
        [12]       -> ["[12]"]
        [12, 15]   -> ["[12]", "[15]"]
        [12-15]    -> ["[12]", "[13]", "[14]", "[15]"]
        [2, 5-7]   -> ["[2]", "[5]", "[6]", "[7]"]
    """
    inner = cluster_text.strip()[1:-1]
    inner = inner.replace("–", "-").replace("—", "-")

    citations = []

    for part in re.split(r"\s*,\s*", inner):
        part = part.strip()
        if not part:
            continue

        range_match = re.fullmatch(r"(\d+)\s*-\s*(\d+)", part)
        if range_match:
            start = int(range_match.group(1))
            end = int(range_match.group(2))

            # Συντηρητικός έλεγχος: μόνο αύξοντα ranges λογικού μεγέθους.
            if start <= end and (end - start) <= 100:
                citations.extend([f"[{n}]" for n in range(start, end + 1)])
            else:
                citations.append(f"[{part}]")
        elif re.fullmatch(r"\d+", part):
            citations.append(f"[{int(part)}]")

    return citations


def _looks_like_author_year_member(member):
    """High-precision validation για ένα author-year citation member."""
    member = str(member).strip()

    if not re.search(r"\b(?:18|19|20)\d{2}[a-z]?\b", member):
        return False

    # Πρέπει πριν από το έτος να υπάρχει τουλάχιστον ένα author-like token.
    year_match = re.search(r"\b(?:18|19|20)\d{2}[a-z]?\b", member)
    before_year = member[:year_match.start()].strip(" ,")

    if not before_year:
        return False

    if re.match(
        r"^(?:Fig\.?|Figure|Table|Section|Eq\.?|Equation|Appendix|Chapter|Volume|Vol\.?)\b",
        before_year,
        re.IGNORECASE,
    ):
        return False

    return bool(re.search(r"[A-Za-zÀ-ÖØ-öø-ÿ]", before_year))


def _find_ieee_groups(sentence):
    """Επιστρέφει non-overlapping IEEE citation groups."""
    groups = []

    ieee_pattern = re.compile(
        r"\[\s*\d+(?:\s*(?:,|[-–—])\s*\d+)*\s*\]"
    )

    for match in ieee_pattern.finditer(sentence):
        members = _expand_ieee_cluster(match.group(0))
        if not members:
            continue

        groups.append({
            "start": match.start(),
            "end": match.end(),
            "style": "IEEE",
            "original": match.group(0),
            "members": members,
        })

    return groups


def _find_narrative_author_year_groups(sentence):
    """
    Εντοπίζει narrative author-year citations όπως:
        Smith (2020)
        Smith and Jones (2020)
        Smith & Jones (2020)
        Smith et al. (2020)
        Smith et al., (2020)
    """
    groups = []

    surname = r"[A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ'’\-]+"

    narrative_pattern = re.compile(
        rf"\b(?:"
        rf"{surname}\s+et\s+al\.?"
        rf"|{surname}\s+(?:and|&)\s+{surname}"
        rf"|{surname}"
        rf")\s*,?\s*\(\s*((?:18|19|20)\d{{2}}[a-z]?)\s*\)"
    )

    for match in narrative_pattern.finditer(sentence):
        groups.append({
            "start": match.start(),
            "end": match.end(),
            "style": "AUTHOR_YEAR_NARRATIVE",
            "original": match.group(0),
            "members": [match.group(0)],
        })

    return groups


def _find_parenthetical_author_year_groups(sentence):
    """
    Εντοπίζει parenthetical author-year clusters.

    Μονή citation:
        (Smith, 2020)

    Πολλαπλές citations με σαφές delimiter ';':
        (Smith, 2020; Brown & Jones, 2021)

    Δεν γίνεται επιθετικό splitting σε αμφίβολες περιπτώσεις χωρίς ';'.
    """
    groups = []

    parenthetical_pattern = re.compile(r"\(([^()]*)\)")

    for match in parenthetical_pattern.finditer(sentence):
        inner = match.group(1).strip()

        if not re.search(r"\b(?:18|19|20)\d{2}[a-z]?\b", inner):
            continue

        raw_members = [m.strip() for m in re.split(r"\s*;\s*", inner)]

        # Όλα τα members πρέπει να μοιάζουν με author-year citations.
        if not raw_members or not all(_looks_like_author_year_member(m) for m in raw_members):
            continue

        # Για Target_Citation κρατάμε citation-like μορφή με παρενθέσεις.
        display_members = [f"({m})" for m in raw_members]

        groups.append({
            "start": match.start(),
            "end": match.end(),
            "style": "AUTHOR_YEAR_PARENTHETICAL",
            "original": match.group(0),
            "members": display_members,
        })

    return groups


def _remove_overlapping_groups(groups):
    """
    Κρατάει non-overlapping citation groups.
    Προτιμά νωρίτερο και μεγαλύτερο span όταν υπάρχει overlap.
    """
    groups = sorted(
        groups,
        key=lambda g: (g["start"], -(g["end"] - g["start"])),
    )

    selected = []

    for group in groups:
        overlaps = any(
            not (group["end"] <= kept["start"] or group["start"] >= kept["end"])
            for kept in selected
        )

        if not overlaps:
            selected.append(group)

    return sorted(selected, key=lambda g: g["start"])


def find_surface_citation_groups(sentence):
    """
    Βρίσκει citation groups μόνο από το surface text.
    Δεν κάνει bibliographic resolution.
    """
    sentence = str(sentence)

    groups = []
    groups.extend(_find_ieee_groups(sentence))
    groups.extend(_find_narrative_author_year_groups(sentence))
    groups.extend(_find_parenthetical_author_year_groups(sentence))

    return _remove_overlapping_groups(groups)


def _render_target_aware_sentence(sentence, groups, target_group_index, target_member_index):
    """
    Αντικαθιστά ΟΛΑ τα citation groups στο context:
      - target citation -> [TARGET_CITATION]
      - κάθε άλλη citation -> [OTHER_CITATION]
    """
    pieces = []
    cursor = 0

    for group_index, group in enumerate(groups):
        pieces.append(sentence[cursor:group["start"]])

        marker_sequence = []
        for member_index, _ in enumerate(group["members"]):
            if (
                group_index == target_group_index
                and member_index == target_member_index
            ):
                marker_sequence.append(TARGET_MARKER)
            else:
                marker_sequence.append(OTHER_MARKER)

        if group["style"] == "AUTHOR_YEAR_PARENTHETICAL":
            replacement = "; ".join(marker_sequence)
        else:
            replacement = " ".join(marker_sequence)

        pieces.append(replacement)
        cursor = group["end"]

    pieces.append(sentence[cursor:])

    marked = "".join(pieces)
    marked = re.sub(r"\s+([,.;:!?])", r"\1", marked)
    marked = re.sub(r"\s{2,}", " ", marked).strip()

    return marked


def create_target_aware_citations(citation_sentences):
    """
    Μετατρέπει citation contexts σε citation-level target-aware instances.

    Επιστρέφει λίστα dictionaries με:
        Original_Sentence
        Target_Citation
        Marked_Sentence

    Μία πρόταση με N citations δημιουργεί N inference instances.
    """
    instances = []
    seen = set()
    unresolved_contexts = []

    for sentence in citation_sentences:
        sentence = str(sentence).strip()
        groups = find_surface_citation_groups(sentence)

        if not groups:
            unresolved_contexts.append(sentence)
            continue

        for group_index, group in enumerate(groups):
            for member_index, target_citation in enumerate(group["members"]):
                marked_sentence = _render_target_aware_sentence(
                    sentence=sentence,
                    groups=groups,
                    target_group_index=group_index,
                    target_member_index=member_index,
                )

                # Safety checks: κάθε inference instance πρέπει να έχει
                # ακριβώς ένα TARGET marker.
                if marked_sentence.count(TARGET_MARKER) != 1:
                    raise ValueError(
                        "Target-aware sentence χωρίς ακριβώς ένα "
                        f"{TARGET_MARKER}: {marked_sentence}"
                    )

                key = (
                    sentence,
                    target_citation,
                    marked_sentence,
                )

                if key in seen:
                    continue

                seen.add(key)

                instances.append({
                    "Original_Sentence": sentence,
                    "Target_Citation": target_citation,
                    "Marked_Sentence": marked_sentence,
                })

    print("\n=== TARGET-AWARE CITATION CONVERSION ===")
    print("Citation contexts πριν το targetization:", len(citation_sentences))
    print("Target-aware citation instances:", len(instances))
    print("Contexts χωρίς αναγνωρίσιμο author-year/IEEE marker:", len(unresolved_contexts))

    if unresolved_contexts:
        print("\nΠαραδείγματα unresolved contexts (έως 5):")
        for example in unresolved_contexts[:5]:
            print("-", example)

    return instances

## 5. Load BERT sentiment component and factorized LLaMA-3-8B function classifier

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER"
PDF_DIR = os.path.join(PROJECT_ROOT, "PDF")

# -------------------------
# BERT sentiment component
# -------------------------
BERT_SENTIMENT_MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "pretrained_citation_polarity_recognition",
    "BERT_A1_BASE_CLINICAL_WEIGHTED_CE_2e-5",
    "saved_model",
)
BERT_SENTIMENT_TOKENIZER_DIR = os.path.join(
    PROJECT_ROOT,
    "pretrained_citation_polarity_recognition",
    "BERT_A1_BASE_CLINICAL_WEIGHTED_CE_2e-5",
    "saved_tokenizer",
)

sentiment_tokenizer = BertTokenizer.from_pretrained(BERT_SENTIMENT_TOKENIZER_DIR)
sentiment_model = BertForSequenceClassification.from_pretrained(BERT_SENTIMENT_MODEL_DIR)
sentiment_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sentiment_model.to(sentiment_device)
sentiment_model.eval()

SENTIMENT_ID_TO_LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}

# -------------------------
# Factorized LLaMA function model
# -------------------------
print("Choose factorized CFA training condition:")
print("1: 70% REAL POOL (ACT2 + FOCAL + MultiCite)")
print("2: 70% REAL POOL (ACT2 + FOCAL + MultiCite) + SYNTHETIC")
DATASET_CHOICE = input("Choice (1/2): ").strip()

if DATASET_CHOICE == "1":
    DATASET_TAG = "REALPOOL70_Train_REALPOOL30_Val_JournalReal_Test"
elif DATASET_CHOICE == "2":
    DATASET_TAG = "REALPOOL70_SYNTHETIC_Train_REALPOOL30_Val_JournalReal_Test"
else:
    raise ValueError("Choice must be 1 or 2.")

LLAMA_BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
FOLDER_MODEL = "LLAMA3_8B"
FOLDER_INPUT = "input_citation"
MODEL_EXPERIMENT_DIR = os.path.join(
    PROJECT_ROOT,
    "pretrained_models_factorized_cfa",
    FOLDER_MODEL,
    FOLDER_INPUT,
    DATASET_TAG,
)
LLAMA_ADAPTER_DIR = os.path.join(MODEL_EXPERIMENT_DIR, "saved_model")
LLAMA_TOKENIZER_DIR = os.path.join(MODEL_EXPERIMENT_DIR, "saved_tokenizer")

if not os.path.isdir(LLAMA_ADAPTER_DIR):
    raise FileNotFoundError(f"Factorized LoRA adapter not found:\n{LLAMA_ADAPTER_DIR}")
if not os.path.isdir(LLAMA_TOKENIZER_DIR):
    raise FileNotFoundError(f"Factorized tokenizer not found:\n{LLAMA_TOKENIZER_DIR}")

LLAMA_COMPUTE_DTYPE = torch.bfloat16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=LLAMA_COMPUTE_DTYPE,
)

llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_TOKENIZER_DIR, use_fast=True)
llama_tokenizer.clean_up_tokenization_spaces = False
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "left"

for marker in ["[TARGET_CITATION]", "[OTHER_CITATION]"]:
    marker_ids = llama_tokenizer.encode(marker, add_special_tokens=False)
    print(marker, "->", llama_tokenizer.tokenize(marker), "->", marker_ids)
    assert len(marker_ids) == 1

# BERT sentiment and LLaMA share the GPU sequentially at inference. Load LLaMA after BERT.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_llama = AutoModelForCausalLM.from_pretrained(
    LLAMA_BASE_MODEL,
    device_map={"": 0},
    quantization_config=bnb_config,
    dtype=LLAMA_COMPUTE_DTYPE,
    trust_remote_code=True,
)
base_llama.resize_token_embeddings(len(llama_tokenizer))
llama_model = PeftModel.from_pretrained(base_llama, LLAMA_ADAPTER_DIR)
llama_model.eval()

print("Factorized CFA model loaded:", MODEL_EXPERIMENT_DIR)

## 6. Four function labels, final 12-class mapping, and inference functions

In [ ]:
FUNCTION_LABELS = [
    "Background_related_work",
    "Comparison",
    "Using_the_work",
    "Extending_the_work",
]

FINAL_LABEL_MAP = {
    ("Positive", "Extending_the_work"): "Constructive_extension",
    ("Positive", "Using_the_work"): "Supportive_application",
    ("Positive", "Background_related_work"): "Aligned_contextualization",
    ("Positive", "Comparison"): "Positive_comparison",
    ("Neutral", "Extending_the_work"): "Neutral_extension",
    ("Neutral", "Using_the_work"): "Neutral_application",
    ("Neutral", "Background_related_work"): "Descriptive_backgrounding",
    ("Neutral", "Comparison"): "Balanced_comparison",
    ("Negative", "Extending_the_work"): "Critical_extension",
    ("Negative", "Using_the_work"): "Misguided_application",
    ("Negative", "Background_related_work"): "Critical_backgrounding",
    ("Negative", "Comparison"): "Negative_comparison",
}

RARE_FINAL_LABELS = {
    "Constructive_extension",
    "Supportive_application",
    "Positive_comparison",
    "Critical_extension",
    "Misguided_application",
    "Negative_comparison",
    "Neutral_extension",
    "Critical_backgrounding",
}

label_token_lengths = {
    label: len(llama_tokenizer(label, add_special_tokens=False)["input_ids"])
    for label in FUNCTION_LABELS
}
GENERATION_MAX_NEW_TOKENS = max(label_token_lengths.values()) + 1
MAX_SEQ_LEN = 256
MAX_PROMPT_LEN = MAX_SEQ_LEN - GENERATION_MAX_NEW_TOKENS
print("Function label token lengths:", label_token_lengths)
print("GENERATION_MAX_NEW_TOKENS:", GENERATION_MAX_NEW_TOKENS)


def predict_sentiment(citation_sentence):
    """BERT sentiment prediction for the same target-aware context."""
    encoded = sentiment_tokenizer(
        [str(citation_sentence)],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    ).to(sentiment_device)
    with torch.inference_mode():
        outputs = sentiment_model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)[0]
        pred_id = int(torch.argmax(probs).item())
    return {
        "sentiment": SENTIMENT_ID_TO_LABEL[pred_id],
        "sentiment_confidence": float(probs.max().item()),
        "prob_negative": float(probs[0].item()),
        "prob_neutral": float(probs[1].item()),
        "prob_positive": float(probs[2].item()),
    }


def build_llama3_messages(citation_sentence):
    system_message = (
        "You are a strict citation-function classification assistant. "
        "You must answer with exactly ONE label from the list."
    )
    user_message = (
        str(citation_sentence).strip()
        + "\n\nClassify the function of [TARGET_CITATION]. "
          "Treat [OTHER_CITATION] only as contextual citations."
        + "\n\nChoose ONE of the following labels:\n"
        + "\n".join(f"- {label}" for label in FUNCTION_LABELS)
        + "\n\nAnswer with ONLY the label name."
    )
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]


def _plain_ids(encoded):
    if isinstance(encoded, dict):
        ids = encoded["input_ids"]
    elif hasattr(encoded, "input_ids"):
        ids = encoded.input_ids
    else:
        ids = encoded
    if torch.is_tensor(ids):
        ids = ids.squeeze(0).tolist()
    if isinstance(ids, (list, tuple)) and ids and isinstance(ids[0], (list, tuple)):
        ids = ids[0]
    return list(ids)


def llama3_eot_token_id():
    eot_id = llama_tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if eot_id is None or eot_id < 0 or eot_id == llama_tokenizer.unk_token_id:
        raise ValueError("Saved tokenizer has no valid <|eot_id|> token.")
    return eot_id


def llama3_terminators():
    ids = [llama_tokenizer.eos_token_id, llama3_eot_token_id()]
    return list(dict.fromkeys(i for i in ids if i is not None))


def centered_inference_window(raw_input_ids, max_tokens):
    if len(raw_input_ids) <= max_tokens:
        return raw_input_ids
    target_id = llama_tokenizer.convert_tokens_to_ids("[TARGET_CITATION]")
    if target_id in raw_input_ids:
        pos = raw_input_ids.index(target_id)
        start = max(0, pos - max_tokens // 2)
        start = min(start, len(raw_input_ids) - max_tokens)
    else:
        start = 0
    return raw_input_ids[start:start + max_tokens]


def build_generation_ids(citation_sentence, max_prompt_len=MAX_PROMPT_LEN):
    citation_sentence = str(citation_sentence).strip()

    def encode(text):
        return _plain_ids(llama_tokenizer.apply_chat_template(
            build_llama3_messages(text),
            tokenize=True,
            add_generation_prompt=True,
        ))

    prompt_ids = encode(citation_sentence)
    if len(prompt_ids) <= max_prompt_len:
        return prompt_ids

    empty_prompt_ids = encode("")
    available = max_prompt_len - len(empty_prompt_ids)
    if available <= 0:
        raise ValueError("Fixed function prompt does not fit in max_prompt_len.")

    raw_ids = llama_tokenizer(citation_sentence, add_special_tokens=False)["input_ids"]
    target_id = llama_tokenizer.convert_tokens_to_ids("[TARGET_CITATION]")
    original_has_target = target_id in raw_ids
    budget = min(available, len(raw_ids))

    while budget > 0:
        trimmed_ids = centered_inference_window(raw_ids, budget)
        if original_has_target and target_id not in trimmed_ids:
            raise ValueError("Inference truncation removed [TARGET_CITATION].")
        trimmed_text = llama_tokenizer.decode(trimmed_ids, skip_special_tokens=False).strip()
        if original_has_target:
            check_ids = llama_tokenizer(trimmed_text, add_special_tokens=False)["input_ids"]
            if target_id not in check_ids:
                budget -= 1
                continue
        prompt_ids = encode(trimmed_text)
        if len(prompt_ids) <= max_prompt_len:
            return prompt_ids
        budget -= 1

    raise ValueError("Could not safely truncate factorized LLaMA inference prompt.")


FUNCTION_PROBABILITY_COLUMNS = {
    "Background_related_work": "Prob_Function_Background",
    "Comparison": "Prob_Function_Comparison",
    "Using_the_work": "Prob_Function_Using",
    "Extending_the_work": "Prob_Function_Extending",
}


def score_function_candidates(prompt_ids):
    """
    Score all four allowed function labels for the SAME prompt.

    Method:
      1. Append each candidate label to the prompt.
      2. Obtain teacher-forced token log-probabilities from the causal LM.
      3. Average token log-probabilities within each label (length normalization).
      4. Softmax the four normalized sequence scores.

    The resulting values are relative candidate-confidence scores over the four
    allowed labels. They are useful for audit/sensitivity analysis but should not
    be interpreted as externally calibrated probabilities.
    """
    candidate_token_ids = {
        label: llama_tokenizer(label, add_special_tokens=False)["input_ids"]
        for label in FUNCTION_LABELS
    }

    if any(len(ids) == 0 for ids in candidate_token_ids.values()):
        raise ValueError("At least one function label tokenized to an empty sequence.")

    sequences = [prompt_ids + candidate_token_ids[label] for label in FUNCTION_LABELS]
    max_len = max(len(seq) for seq in sequences)
    pad_id = llama_tokenizer.pad_token_id
    if pad_id is None:
        pad_id = llama_tokenizer.eos_token_id

    batch_input_ids = torch.full(
        (len(sequences), max_len),
        fill_value=pad_id,
        dtype=torch.long,
        device=llama_model.device,
    )
    batch_attention_mask = torch.zeros(
        (len(sequences), max_len),
        dtype=torch.long,
        device=llama_model.device,
    )

    for row_idx, seq in enumerate(sequences):
        seq_tensor = torch.tensor(seq, dtype=torch.long, device=llama_model.device)
        batch_input_ids[row_idx, :len(seq)] = seq_tensor
        batch_attention_mask[row_idx, :len(seq)] = 1

    with torch.inference_mode():
        outputs = llama_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            use_cache=False,
        )

    # Gather only the logits needed to score candidate tokens. This avoids
    # materializing a full-vocabulary log-softmax for every sequence position.
    row_indices = []
    position_indices = []
    target_token_ids = []
    token_slices = {}
    cursor = 0
    prompt_len = len(prompt_ids)

    for row_idx, label in enumerate(FUNCTION_LABELS):
        label_ids = candidate_token_ids[label]
        start = cursor
        for token_offset, token_id in enumerate(label_ids):
            # Token at sequence position (prompt_len + token_offset) is predicted
            # by logits from the immediately preceding position.
            row_indices.append(row_idx)
            position_indices.append(prompt_len + token_offset - 1)
            target_token_ids.append(token_id)
            cursor += 1
        token_slices[label] = (start, cursor)

    row_indices = torch.tensor(row_indices, dtype=torch.long, device=llama_model.device)
    position_indices = torch.tensor(position_indices, dtype=torch.long, device=llama_model.device)
    target_token_ids = torch.tensor(target_token_ids, dtype=torch.long, device=llama_model.device)

    selected_logits = outputs.logits[row_indices, position_indices, :].float()
    selected_log_probs = torch.log_softmax(selected_logits, dim=-1)
    token_log_probs = selected_log_probs.gather(
        1,
        target_token_ids.unsqueeze(1),
    ).squeeze(1)

    mean_log_probs = []
    for label in FUNCTION_LABELS:
        start, end = token_slices[label]
        mean_log_probs.append(token_log_probs[start:end].mean())

    mean_log_probs = torch.stack(mean_log_probs)
    candidate_probs = torch.softmax(mean_log_probs, dim=0).detach().cpu().numpy()

    probs = {
        label: float(candidate_probs[i])
        for i, label in enumerate(FUNCTION_LABELS)
    }

    del outputs, selected_logits, selected_log_probs, token_log_probs, mean_log_probs
    return probs


def predict_function(citation_sentence):
    """
    Preserve the original deterministic generation-based function prediction,
    and add candidate-label confidence diagnostics without changing Function_Label.
    """
    prompt_ids = build_generation_ids(citation_sentence)
    inputs = {
        "input_ids": torch.tensor([prompt_ids], dtype=torch.long, device=llama_model.device),
        "attention_mask": torch.ones((1, len(prompt_ids)), dtype=torch.long, device=llama_model.device),
    }

    # Original function prediction: deterministic generation.
    with torch.inference_mode():
        output = llama_model.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=llama3_terminators(),
            pad_token_id=llama_tokenizer.pad_token_id,
        )

    generated_ids = output[:, inputs["input_ids"].shape[1]:]
    pred = llama_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    pred = pred.strip().split("\n")[0].strip(" .,:;")
    function_label = pred if pred in FUNCTION_LABELS else "unknown"

    # Additional diagnostic scoring over all four allowed labels.
    function_probs = score_function_candidates(prompt_ids)
    ranked = sorted(function_probs.items(), key=lambda kv: kv[1], reverse=True)
    scored_top_label = ranked[0][0]
    function_margin = float(ranked[0][1] - ranked[1][1])

    # Confidence is tied to the ACTUAL generation-based Function_Label.
    # If generation produced an invalid label, confidence remains NaN rather than
    # silently replacing the prediction and changing the original pipeline.
    function_confidence = (
        float(function_probs[function_label])
        if function_label in function_probs
        else np.nan
    )

    return {
        "function_label": function_label,
        "function_confidence": function_confidence,
        "prob_function_background": float(function_probs["Background_related_work"]),
        "prob_function_comparison": float(function_probs["Comparison"]),
        "prob_function_using": float(function_probs["Using_the_work"]),
        "prob_function_extending": float(function_probs["Extending_the_work"]),
        "function_margin": function_margin,
        "function_scored_top_label": scored_top_label,
        "function_score_agreement": bool(function_label == scored_top_label),
    }


def final_cfa_label(sentiment, function_label):
    return FINAL_LABEL_MAP.get((sentiment, function_label), "unknown")

## 7. Process PDFs and write factorized rare-citation outputs

In [ ]:
pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])

summary_path = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER/rare_citations_summary.csv"

details_dir = os.path.join(PROJECT_ROOT, "rare_citations_pdf_details_factorized_cfa")
os.makedirs(details_dir, exist_ok=True)

REPROCESS_DETAILS_MISSING_FUNCTION_CONFIDENCE = True

REQUIRED_FUNCTION_CONFIDENCE_COLUMNS = {
    "Function_Confidence",
    "Prob_Function_Background",
    "Prob_Function_Comparison",
    "Prob_Function_Using",
    "Prob_Function_Extending",
    "Function_Margin",
    "Function_Scored_Top_Label",
    "Function_Score_Agreement",
}

if os.path.exists(summary_path):
    existing_df = pd.read_csv(summary_path)
    processed_files = set(existing_df["filename"].astype(str).unique())
else:
    existing_df = pd.DataFrame()
    processed_files = set()


def detail_file_has_function_confidence(pdf_file):
    pdf_stem = os.path.splitext(pdf_file)[0]
    detail_path = os.path.join(details_dir, f"{pdf_stem}_factorized_cfa_details.csv")
    if not os.path.exists(detail_path):
        return False
    try:
        detail_cols = set(pd.read_csv(detail_path, nrows=0).columns)
        return REQUIRED_FUNCTION_CONFIDENCE_COLUMNS.issubset(detail_cols)
    except Exception:
        return False


files_to_process = []
for f in pdf_files:
    is_new = f not in processed_files
    needs_confidence_refresh = (
        REPROCESS_DETAILS_MISSING_FUNCTION_CONFIDENCE
        and not detail_file_has_function_confidence(f)
    )
    if is_new or needs_confidence_refresh:
        files_to_process.append(f)

print("Total PDFs:", len(pdf_files))
print("Already listed in summary:", len(processed_files))
print("PDFs to process/reprocess:", len(files_to_process))
if REPROCESS_DETAILS_MISSING_FUNCTION_CONFIDENCE:
    print("Legacy detail files missing function-confidence columns will be refreshed.")

pdf_level_stats = []
last_results_df = None
last_pdf_name = None

for pdf_file in files_to_process:
    print(f"\nProcessing: {pdf_file}")
    pdf_path = os.path.join(PDF_DIR, pdf_file)

    citation_sentences = main(pdf_path)
    target_aware_citations = create_target_aware_citations(citation_sentences)

    rows = []
    for item in target_aware_citations:
        marked_sentence = item["Marked_Sentence"]

        # Factor 1: BERT sentiment on the same target-aware context.
        sent = predict_sentiment(marked_sentence)

        # Factor 2: LLaMA function on the same target-aware context.
        # Function_Label remains generation-based; candidate scoring adds
        # confidence/probability diagnostics without changing the prediction.
        func = predict_function(marked_sentence)
        function_label = func["function_label"]

        # Deterministic 3 × 4 -> 12 final CFA label.
        final_label = final_cfa_label(sent["sentiment"], function_label)
        is_rare = 1 if final_label in RARE_FINAL_LABELS else 0

        rows.append({
            "FileName": pdf_file,
            "Original_Sentence": item["Original_Sentence"],
            "Target_Citation": item["Target_Citation"],
            "Marked_Sentence": marked_sentence,
            "Sentiment": sent["sentiment"],
            "Sentiment_Confidence": sent["sentiment_confidence"],
            "Prob_Negative": sent["prob_negative"],
            "Prob_Neutral": sent["prob_neutral"],
            "Prob_Positive": sent["prob_positive"],
            "Function_Label": function_label,
            "Function_Confidence": func["function_confidence"],
            "Prob_Function_Background": func["prob_function_background"],
            "Prob_Function_Comparison": func["prob_function_comparison"],
            "Prob_Function_Using": func["prob_function_using"],
            "Prob_Function_Extending": func["prob_function_extending"],
            "Function_Margin": func["function_margin"],
            "Function_Scored_Top_Label": func["function_scored_top_label"],
            "Function_Score_Agreement": func["function_score_agreement"],
            # Preserve the old downstream column name for the final 12-class label.
            "Explanation_Label": final_label,
            "Is_Rare_Citation": is_rare,
        })

    df_pdf = pd.DataFrame(rows, columns=[
        "FileName",
        "Original_Sentence",
        "Target_Citation",
        "Marked_Sentence",
        "Sentiment",
        "Sentiment_Confidence",
        "Prob_Negative",
        "Prob_Neutral",
        "Prob_Positive",
        "Function_Label",
        "Function_Confidence",
        "Prob_Function_Background",
        "Prob_Function_Comparison",
        "Prob_Function_Using",
        "Prob_Function_Extending",
        "Function_Margin",
        "Function_Scored_Top_Label",
        "Function_Score_Agreement",
        "Explanation_Label",
        "Is_Rare_Citation",
    ])

    print(df_pdf.head())
    print("Target citation instances:", len(df_pdf))
    if len(df_pdf):
        print("Unknown function predictions:", int((df_pdf["Function_Label"] == "unknown").sum()))
        print("Unknown final labels:", int((df_pdf["Explanation_Label"] == "unknown").sum()))
        print(
            "Generation/scored-top disagreements:",
            int((~df_pdf["Function_Score_Agreement"].astype(bool)).sum()),
        )
        print("Mean Function_Confidence:", float(df_pdf["Function_Confidence"].mean()))
        print("Mean Function_Margin:", float(df_pdf["Function_Margin"].mean()))

    pdf_stem = os.path.splitext(pdf_file)[0]
    detail_path = os.path.join(details_dir, f"{pdf_stem}_factorized_cfa_details.csv")
    df_pdf.to_csv(detail_path, index=False, encoding="utf-8-sig")
    print("Saved details:", detail_path)

    if len(df_pdf) > 0:
        rare_percentage = df_pdf["Is_Rare_Citation"].mean()
        total_found = len(df_pdf)
        unknown_count = int((df_pdf["Explanation_Label"] == "unknown").sum())
    else:
        rare_percentage = np.nan
        total_found = 0
        unknown_count = 0

    pdf_level_stats.append({
        "filename": pdf_file,
        "less_frequent_citation_ratio": rare_percentage,
        "total_citations_count": total_found
    })

    last_results_df = df_pdf
    last_pdf_name = pdf_file

print("\nDataFrame τελευταίου PDF:")
print(last_results_df)

summary_df = pd.DataFrame(pdf_level_stats)

# Update the summary without creating duplicate filename rows when an older PDF
# is reprocessed only to add the new function-confidence diagnostics.
if len(summary_df) > 0:
    if os.path.exists(summary_path):
        existing_summary = pd.read_csv(summary_path)
        existing_summary["filename"] = existing_summary["filename"].astype(str)
        refreshed_names = set(summary_df["filename"].astype(str))
        existing_summary = existing_summary[
            ~existing_summary["filename"].isin(refreshed_names)
        ].copy()
        combined_summary = pd.concat(
            [existing_summary, summary_df],
            ignore_index=True,
            sort=False,
        )
    else:
        combined_summary = summary_df.copy()

    combined_summary.to_csv(summary_path, index=False)
else:
    combined_summary = pd.read_csv(summary_path) if os.path.exists(summary_path) else summary_df

print("\nΣύνοψη σπάνιων citations ανά PDF (τρέχον run):")
print(summary_df)
print("Συνολικές εγγραφές στο rare_citations_summary.csv:", len(combined_summary))

last_results_df